In [1]:
import rasterio
import pandas as pd


districts_77_coords = pd.concat([
    pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\nepal_70_districts_coords.csv')[['district','lat','lon']],
    pd.DataFrame([
        {'district': 'Kathmandu', 'lat': 27.7172, 'lon': 85.3240},
        {'district': 'Kaski',     'lat': 28.2096, 'lon': 83.9856},
        {'district': 'Ilam',      'lat': 26.9126, 'lon': 87.9296},
        {'district': 'Manang',    'lat': 28.6667, 'lon': 84.0167},
        {'district': 'Morang',    'lat': 26.4525, 'lon': 87.2718},
        {'district': 'Banke',     'lat': 28.0503, 'lon': 81.6158},
        {'district': 'Dhanusa',   'lat': 26.7288, 'lon': 85.9286},
    ])
], ignore_index=True)

SLOPE_FILE = r"C:\Nepal_Flood_Project\Data\viz.SRTMGL1_slope.tif"

slopes = {}
with rasterio.open(SLOPE_FILE) as src:
    for _, row in districts_77_coords.iterrows():
        for val in src.sample([(row['lon'], row['lat'])]):
            slopes[row['district']] = round(float(val[0]), 2)

print(f'Slope extracted for {len(slopes)} / 77 locations')
for d, s in slopes.items():
    print(f'{d:20} -> {s}°')

Slope extracted for 77 / 77 locations
Achham               -> 9.74°
Arghakhanchi         -> 11.65°
Baglung              -> 3.84°
Baitadi              -> 6.35°
Bajhang              -> 10.19°
Bajura               -> 14.79°
Bara                 -> 1.67°
Bardiya              -> 1.47°
Bhaktapur            -> 6.93°
Chitwan              -> 1.64°
Dadeldhura           -> 3.99°
Dailekh              -> 14.59°
Dang                 -> 1.04°
Darchula             -> 25.75°
Dhading              -> 2.64°
Dhankuta             -> 6.11°
Dolakha              -> 18.37°
Dolpa                -> 7.18°
Doti                 -> 15.88°
Gorkha               -> 8.31°
Gulmi                -> 3.6°
Humla                -> 9.96°
Jhapa                -> 1.31°
Jumla                -> 5.4°
Kalikot              -> 11.55°
Kanchanpur           -> 4.87°
Kavrepalanchok       -> 2.64°
Khotang              -> 11.96°
Lalitpur             -> 4.26°
Lamjung              -> 5.82°
Mahottari            -> 3.24°
Mugu                 -> 1

In [2]:
import json
with open(r'C:\Nepal_Flood_Project\Data\Districts_77\slope_lookup_77.json', 'w') as f:
    json.dump(slopes, f)
print('Saved!')

Saved!


In [3]:
# Step 1: Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os
import re

print(' All libraries imported!')

 All libraries imported!


In [4]:
# Step 2: Setting File Paths

NASA_CLIMATE_FOLDER = r"C:\Nepal_Flood_Project\Data\Nasa_Data"
NASA_SOIL_FOLDER     = r"C:\Nepal_Flood_Project\Data\landslide_data\nasapower_data"
DISTRICTS_70_FOLDER  = r"C:\Nepal_Flood_Project\Data\Districts_77"
EMDAT_FILE   = r"C:\Nepal_Flood_Project\Data\landslide_data\landslideeventEMDAT.xlsx"
BIPAD_LANDSLIDE_FILE = r"C:\Nepal_Flood_Project\Data\landslide_data\bipad_landslide_nepal.csv"  
OUTPUT_FILE  = r'C:\Nepal_Flood_Project\Data\final_landslide_dataset_77districts.csv'


PLACE_TO_DISTRICT = {
    'kathmandu': 'Kathmandu', 'pokhara': 'Kaski', 'illam': 'Ilam',
    'manang': 'Manang', 'biratnagar': 'Morang', 'nepalgunj': 'Banke', 'janakpur': 'Dhanusa',
}
CLIMATE_FILES = {'kathmandu':'ktmnasa.csv','pokhara':'pokhara.csv','illam':'illam.csv',
                  'manang':'manangnasa.csv','biratnagar':'biratnagar.csv','nepalgunj':'nepalgunj.csv','janakpur':'janakpur.csv'}
SOIL_FILES = {'kathmandu':'kathmandu_soilmoisture.csv','pokhara':'pokhara_soilmoisture.csv',
              'biratnagar':'biratnagar_soilmoisture.csv','janakpur':'janakpur_soilmoisture.csv',
              'nepalgunj':'nepalgunj_soilmoisture.csv','illam':'illam_soilmoisture.csv','manang':'manang_soilmoisture.csv'}


terrain_lookup_70 = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\nepal_70_districts_coords.csv').set_index('district')['terrain'].to_dict()
TERRAIN_MAP_7 = {'Kathmandu':'Hilly','Kaski':'Hilly','Ilam':'Hilly','Manang':'Mountain','Morang':'Terai','Banke':'Terai','Dhanusa':'Terai'}

CITY_SLOPES = slopes  

In [5]:
print('Paths and mappings set!')
print('CITY_SLOPES has', len(CITY_SLOPES), 'entries')
print('terrain_lookup_70 has', len(terrain_lookup_70), 'entries')

Paths and mappings set!
CITY_SLOPES has 77 entries
terrain_lookup_70 has 70 entries


In [7]:
# Step 3: Reading NASA POWER Climate Files
OLD_FILENAME_LOOKUP = {
    'Rukum East': 'Eastern_Rukum',
    'Kapilbastu': 'Kapilvastu',
    'Nawalparasi East': 'Nawalpur',
    'Tanahu': 'Tanahun',
}

districts_70_data = []
for district in terrain_lookup_70.keys():
    file_key = OLD_FILENAME_LOOKUP.get(district, district.replace(" ", "_"))
    filepath = os.path.join(DISTRICTS_70_FOLDER, f'nasa_{file_key}.csv')
    if os.path.exists(filepath):
        df_d = read_nasa_file(filepath, district)  
        df_d = df_d.rename(columns={'GWETROOT': 'soil_moisture'})
        districts_70_data.append(df_d)
    else:
        print(f'STILL MISSING: {district} (tried {filepath})')

df_70districts = pd.concat(districts_70_data, ignore_index=True)
df_70districts['terrain'] = df_70districts['location'].map(terrain_lookup_70)

df = pd.concat([df_7places, df_70districts], ignore_index=True)
df = df.drop(columns=['date_key'], errors='ignore')
print(f'Combined: {df.shape[0]} rows, {df["location"].nunique()} locations')

Combined: 731269 rows, 77 locations


In [8]:
# Step 4: Converting DOY to Date + Add Season

df['date'] = pd.to_datetime(
    df['YEAR'].astype(str) + '-' + df['DOY'].astype(str),
    format='%Y-%j'
)
df['month'] = df['date'].dt.month
df['year']  = df['date'].dt.year
df['day']   = df['date'].dt.day

def get_season(month):
    if month in [3, 4, 5]:   return 'pre_monsoon'
    elif month in [6, 7, 8, 9]: return 'monsoon'
    elif month in [10, 11]:  return 'post_monsoon'
    else:                    return 'winter'

df['season'] = df['month'].apply(get_season)

df = df.rename(columns={
    'T2M':         'temperature',
    'PRECTOTCORR': 'rainfall',
    'RH2M':        'humidity',
    'WS2M':        'wind_speed'
})

df = df.drop(columns=['YEAR', 'DOY'], errors='ignore')
df = df.sort_values(['location', 'date']).reset_index(drop=True)

print('Date conversion done!')
print(df[['date', 'month', 'season', 'location', 'temperature', 'rainfall', 'soil_moisture']].head(5))

Date conversion done!
        date  month  season location  temperature  rainfall  soil_moisture
0 2000-01-01      1  winter   Achham        10.89       0.0           0.38
1 2000-01-02      1  winter   Achham        11.75       0.0           0.38
2 2000-01-03      1  winter   Achham        11.48       0.0           0.38
3 2000-01-04      1  winter   Achham        12.01       0.0           0.38
4 2000-01-05      1  winter   Achham        11.95       0.0           0.38


In [10]:
# Step 5: Adding Rolling Rainfall Features

df['rain_3day'] = (
    df.groupby('location')['rainfall']
    .transform(lambda x: x.rolling(window=3, min_periods=1).sum())
)
df['rain_7day'] = (
    df.groupby('location')['rainfall']
    .transform(lambda x: x.rolling(window=7, min_periods=1).sum())
)
print('Rolling rainfall calculated!')

Rolling rainfall calculated!


In [11]:
# Step 7: Adding Slope Feature

df['slope'] = df['location'].map(CITY_SLOPES)

print('Slope feature added!')
print('Missing slope values:', df['slope'].isna().sum())
print(df.groupby('location')['slope'].mean().round(2))

Slope feature added!
Missing slope values: 0
location
Achham           9.74
Arghakhanchi    11.65
Baglung          3.84
Baitadi          6.35
Bajhang         10.19
                ...  
Syangja          5.56
Tanahu           3.09
Taplejung        9.38
Terhathum        9.90
Udayapur         1.04
Name: slope, Length: 77, dtype: float64


In [12]:
# Step 8
bipad_ls = pd.read_csv(r"C:\Nepal_Flood_Project\Data\bipad_landslide_nepal_2011-2026.csv", low_memory=False)
bipad_ls = bipad_ls[bipad_ls['Hazard'] == 'Landslide'].copy()
bipad_ls['incident_date'] = pd.to_datetime(bipad_ls['Incident on']).dt.date

all_locations = df['location'].unique()
ls_high_dates = {}
ls_medium_dates = {}

for loc in all_locations:
    dates = set(bipad_ls.loc[bipad_ls['District'] == loc, 'incident_date'])
    ls_high_dates[loc] = dates
    med = set()
    for d in dates:
        for days_before in range(1, 15):
            pre = d - pd.Timedelta(days=days_before)
            if pre not in dates:
                med.add(pre)
    ls_medium_dates[loc] = med
    print(f'{loc:20} High days: {len(dates):4} | Medium spillover: {len(med)}')

def assign_landslide_risk(row):
    row_date = pd.Timestamp(row['date']).date()
    loc = row['location']
    terrain = row['terrain']
    slope = row['slope']

    if terrain == 'Terai':
        return 'Low'

    if row_date in ls_high_dates.get(loc, set()):
        return 'High' if slope >= 10 else 'Medium'
    if row_date in ls_medium_dates.get(loc, set()):
        return 'Medium' if slope >= 1 else 'Low'
    return 'Low'

df['landslide_risk'] = df.apply(assign_landslide_risk, axis=1)
df['landslide_risk_label'] = df['landslide_risk'].map({'Low':0,'Medium':1,'High':2})

print('\nLabel distribution:')
print(df['landslide_risk'].value_counts())
print(df['landslide_risk'].value_counts(normalize=True).mul(100).round(2))

Achham               High days:   27 | Medium spillover: 327
Arghakhanchi         High days:   41 | Medium spillover: 372
Baglung              High days:  102 | Medium spillover: 879
Baitadi              High days:   65 | Medium spillover: 456
Bajhang              High days:   56 | Medium spillover: 510
Bajura               High days:   66 | Medium spillover: 649
Banke                High days:    2 | Medium spillover: 28
Bara                 High days:    0 | Medium spillover: 0
Bardiya              High days:    2 | Medium spillover: 28
Bhaktapur            High days:   30 | Medium spillover: 316
Bhojpur              High days:   82 | Medium spillover: 583
Chitwan              High days:   78 | Medium spillover: 611
Dadeldhura           High days:   29 | Medium spillover: 282
Dailekh              High days:   57 | Medium spillover: 508
Dang                 High days:   23 | Medium spillover: 255
Darchula             High days:   76 | Medium spillover: 669
Dhading              High da

In [13]:
# Step 9: Encoding Categorical Columns

from sklearn.preprocessing import LabelEncoder

le_location = LabelEncoder()
df['location_encoded'] = le_location.fit_transform(df['location'])

le_terrain = LabelEncoder()
df['terrain_encoded'] = le_terrain.fit_transform(df['terrain'])

le_season = LabelEncoder()
df['season_encoded'] = le_season.fit_transform(df['season'])

print('Encoding done!')
print('Locations encoded:', len(le_location.classes_))
print('Terrain classes:', le_terrain.classes_)

Encoding done!
Locations encoded: 77
Terrain classes: ['Hilly' 'Mountain' 'Terai']


In [14]:
# Step 10: Handling Missing Values

df = df.replace(-999, np.nan)

print('Missing values before filling:')
print(df.isnull().sum())

numeric_cols = ['temperature', 'rainfall', 'humidity', 'wind_speed',
                 'rain_3day', 'rain_7day', 'soil_moisture', 'slope']
for col in numeric_cols:
    missing = df[col].isnull().sum()
    if missing > 0:
        df[col] = df[col].fillna(df[col].mean())
        print(f'Filled {missing} missing in {col}')

print('\nRemaining missing:', df.isnull().sum().sum())

Missing values before filling:
temperature             0
rainfall                0
humidity                0
wind_speed              0
location                0
soil_moisture           0
terrain                 0
date                    0
month                   0
year                    0
day                     0
season                  0
rain_3day               0
rain_7day               0
slope                   0
landslide_risk          0
landslide_risk_label    0
location_encoded        0
terrain_encoded         0
season_encoded          0
dtype: int64

Remaining missing: 0


In [15]:
# Step 11: Saving Final Dataset
OUTPUT_FILE = r'C:\Nepal_Flood_Project\Data\final_landslide_dataset_77districts.csv'
df.to_csv(OUTPUT_FILE, index=False)
print(f'Full dataset saved: {OUTPUT_FILE}')

ml_features = [
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'rain_3day', 'rain_7day', 'soil_moisture', 'slope',
    'month', 'terrain_encoded', 'location_encoded',
    'landslide_risk_label'
]

df_ml = df[ml_features]
ml_output = OUTPUT_FILE.replace('.csv', '_ml_ready.csv')
df_ml.to_csv(ml_output, index=False)

print(f'ML-ready dataset saved: {ml_output}')
print(f'\nFinal dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'ML-ready: {df_ml.shape[0]} rows x {df_ml.shape[1]} columns')
print('\nFinal label distribution:')
print(df['landslide_risk'].value_counts())
print('\nLandslide data pipeline complete!')

Full dataset saved: C:\Nepal_Flood_Project\Data\final_landslide_dataset_77districts.csv
ML-ready dataset saved: C:\Nepal_Flood_Project\Data\final_landslide_dataset_77districts_ml_ready.csv

Final dataset: 731269 rows x 20 columns
ML-ready: 731269 rows x 12 columns

Final label distribution:
landslide_risk
Low       698223
Medium     31484
High        1562
Name: count, dtype: int64

Landslide data pipeline complete!
